In [1]:
import numpy as np


def simulate_pv_power(duty_cycle: float, irradiance: float = 850.0, temp_c: float = 30.0) -> float:
    """
    Simulates non-linear PV array output power (in Watts) for a given duty cycle in [0.05, 0.95],
    including a partial-shading local peak so simple hill-climbing can get stuck.
    """
    d = float(np.clip(duty_cycle, 0.05, 0.95))
    irr_factor = irradiance / 1000.0
    temp_loss = max(0.0, 1.0 - 0.004 * (temp_c - 25.0))

    # Global peak around d = 0.62 + secondary shading peak around d = 0.28
    global_peak = 250.0 * irr_factor * temp_loss * np.exp(-((d - 0.62) / 0.14) ** 2)
    shading_peak = 145.0 * irr_factor * temp_loss * np.exp(-((d - 0.28) / 0.06) ** 2)
    ripple = 1.5 * np.sin(25.0 * d)
    return float(max(0.0, global_peak + shading_peak + ripple))


def gwo_mppt(n_wolves: int = 12, n_iterations: int = 10, irradiance: float = 850.0, seed: int = 42):
    rng = np.random.default_rng(seed)
    lb, ub = 0.05, 0.95

    # Position represents converter duty cycle (1D optimization problem)
    wolves = rng.uniform(lb, ub, size=(n_wolves, 1))
    # Maximize power => minimize negative power
    losses = np.array([-simulate_pv_power(w[0], irradiance=irradiance) for w in wolves])

    idx = np.argsort(losses)
    alpha_pos, alpha_loss = wolves[idx[0]].copy(), float(losses[idx[0]])
    beta_pos, beta_loss = wolves[idx[1]].copy(), float(losses[idx[1]])
    delta_pos, delta_loss = wolves[idx[2]].copy(), float(losses[idx[2]])

    print("OUTPUT:")
    for it in range(1, n_iterations + 1):
        a = 2.0 - 2.0 * (it / n_iterations)

        for i in range(n_wolves):
            for d in range(1):
                r1, r2 = rng.random(), rng.random()
                A1, C1 = 2.0 * a * r1 - a, 2.0 * r2
                X1 = alpha_pos[d] - A1 * abs(C1 * alpha_pos[d] - wolves[i, d])

                r1, r2 = rng.random(), rng.random()
                A2, C2 = 2.0 * a * r1 - a, 2.0 * r2
                X2 = beta_pos[d] - A2 * abs(C2 * beta_pos[d] - wolves[i, d])

                r1, r2 = rng.random(), rng.random()
                A3, C3 = 2.0 * a * r1 - a, 2.0 * r2
                X3 = delta_pos[d] - A3 * abs(C3 * delta_pos[d] - wolves[i, d])

                wolves[i, d] = np.clip((X1 + X2 + X3) / 3.0, lb, ub)

            loss = -simulate_pv_power(wolves[i, 0], irradiance=irradiance)
            losses[i] = loss

            if loss < alpha_loss:
                delta_loss, delta_pos = beta_loss, beta_pos.copy()
                beta_loss, beta_pos = alpha_loss, alpha_pos.copy()
                alpha_loss, alpha_pos = float(loss), wolves[i].copy()
            elif loss < beta_loss:
                delta_loss, delta_pos = beta_loss, beta_pos.copy()
                beta_loss, beta_pos = float(loss), wolves[i].copy()
            elif loss < delta_loss:
                delta_loss, delta_pos = float(loss), wolves[i].copy()

        print(f"Iteration {it}/{n_iterations} - Best Power (W) : {-alpha_loss:.2f}")

    best_duty = float(alpha_pos[0])
    best_power = -alpha_loss
    naive_power = simulate_pv_power(0.28, irradiance=irradiance)  # Trapped at shading local peak

    print("\n-------- Optimization Results --------")
    print(f"Irradiance (W/m^2)      : {irradiance:.1f}")
    print(f"Trapped Peak Power (W)  : {naive_power:.2f}")
    print(f"Optimal Duty Cycle      : {best_duty:.4f}")
    print(f"Max Extracted Power (W) : {best_power:.2f}")


if __name__ == "__main__":
    gwo_mppt()

OUTPUT:
Iteration 1/10 - Best Power (W) : 204.63
Iteration 2/10 - Best Power (W) : 204.63
Iteration 3/10 - Best Power (W) : 204.63
Iteration 4/10 - Best Power (W) : 208.07
Iteration 5/10 - Best Power (W) : 208.07
Iteration 6/10 - Best Power (W) : 208.07
Iteration 7/10 - Best Power (W) : 208.54
Iteration 8/10 - Best Power (W) : 208.54
Iteration 9/10 - Best Power (W) : 208.54
Iteration 10/10 - Best Power (W) : 208.59

-------- Optimization Results --------
Irradiance (W/m^2)      : 850.0
Trapped Peak Power (W)  : 122.34
Optimal Duty Cycle      : 0.6183
Max Extracted Power (W) : 208.59
